# A3.4 · Budgets and stop conditions

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.3 · Egress control](https://spbreed.github.io/cyber-commons/lessons/A3.3.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A budget is what makes "autonomous" a bounded word. Without one, the honest description of the worst case is "until someone notices", and nobody signs off on that when it is written down.

> **At CyberTravels.** What makes “CyberTravels runs autonomously” a bounded sentence: a ceiling on tokens, wall clock, spend and, above all, on how many refunds one run may issue. R1.

## 2 · The framework

```
   bounded run
   +-------------------------------------------+
   | tokens   <= 60k     wall clock <= 5 min   |
   | steps    <= 25      spend     <= $2.00    |
   | actions  <= 3 writes, 0 deletes           |
   +-------------------------------------------+
              |
     hit any ceiling -> stop, report, hand back

   "autonomous" now has a worst case you can write on a page
```

**Mitigates: T4 Resource Overload · T10 Overwhelming Human-in-the-Loop.**

A budget is what makes "autonomous" a bounded word.

A1.13's loop had no exit condition, so it ran until something outside it
intervened. A ceiling turns that into a defined worst case — and a worst case is
the thing you can actually put in a design document, an incident plan or a risk
register.

Four ceilings, because they bound different failures:

**Tokens or cost.** The visible one. Bounds the bill.

**Wall-clock time.** Bounds a workflow step that never returns.

**Actions.** Bounds *consequence*, and it is the one that matters for security.
Twenty tool calls is a very different blast radius from two thousand, whatever
either costs.

**Downstream calls per target.** Bounds harm to other people. A1.13's damage was
not the token spend, it was the capacity taken from everybody else.

Two design rules:

**Terminate, do not degrade.** A loop that hits a ceiling and keeps going with a
smaller model or a shorter context has not been bounded, it has been redirected.

**Make the ceiling visible in the output.** `stopped_by: action_budget` is a
signal to a human that this run is incomplete. Silent truncation is how a
partial result becomes a reported success, which is A1.16 arriving through a
different door.

> **What this control closes.**
>
> Turns an unbounded loop into a defined worst case, and bounds the harm to **other people's** capacity, not just your bill.

## 3 · The control

In [ ]:
class Budget:
    def __init__(self, tokens=50_000, seconds=60, actions=20, per_target=5):
        self.limits = {"tokens": tokens, "seconds": seconds,
                       "actions": actions, "per_target": per_target}
        self.used = {"tokens": 0, "seconds": 0, "actions": 0}
        self.targets = {}
    def spend(self, tokens=0, seconds=0, action=None, target=None):
        self.used["tokens"] += tokens
        self.used["seconds"] += seconds
        if action: self.used["actions"] += 1
        if target:
            self.targets[target] = self.targets.get(target, 0) + 1
            if self.targets[target] > self.limits["per_target"]:
                return False, f"per_target ({target})"
        for k in ("tokens", "seconds", "actions"):
            if self.used[k] > self.limits[k]:
                return False, k
        return True, None

def loop(budget):
    """A task that cannot succeed - A1.13's exact scenario, now bounded."""
    steps = 0
    while True:
        steps += 1
        ok, hit = budget.spend(tokens=1800, seconds=0.4, action="query",
                               target="reports-db")
        if not ok:
            return {"steps": steps, "stopped_by": hit, "complete": False}

b = Budget()
r = loop(b)
print(f"steps taken : {r['steps']}")
print(f"stopped by  : {r['stopped_by']}")
print(f"complete    : {r['complete']}   <- visible in the output, not silent")
print()
print(f"{'ceiling':14s}{'limit':>9}{'used':>9}")
for k in ("tokens", "seconds", "actions"):
    print(f"{k:14s}{b.limits[k]:>9}{round(b.used[k], 1):>9}")
print(f"{'per_target':14s}{b.limits['per_target']:>9}{b.targets['reports-db']:>9}")
print()
print("per_target fired first, at 6 calls - long before the token budget or the")
print("action budget. That is the ceiling that protects everyone else, and it is")
print("the one most budgets do not have.")
print()
print("`complete: False` is the other half. A run that stops silently and")
print("reports what it managed becomes A1.16 with extra steps.")
assert r["stopped_by"].startswith("per_target") and not r["complete"]

## What you just proved

The impossible task from A1.13 now stops after six steps, halted by the per-target ceiling — before the token or action budgets are anywhere near exhausted — and the result carries `complete: False` rather than reporting what it managed.

## Your turn

Check whether your agent's budget bounds calls per downstream target. If it only bounds tokens, your cost is protected and the service your agent hammers is not.

---

**Next → [A3.5 · Validating what comes back](https://spbreed.github.io/cyber-commons/lessons/A3.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*